## a) Basic setup
1. python packages
2. API keys

In [ ]:
!pip install -q biopython requests pandas dotenv

In [ ]:
import os

# Detect environment and load secrets
try:
    from google.colab import userdata

    # --- Google Colab ---
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    ENTREZ_EMAIL = userdata.get("ENTREZ_EMAIL")
    NCBI_API_KEY = userdata.get("NCBI_API_KEY")

    print("Running in Google Colab.")

except ImportError:
    # --- Local Jupyter ---
    from dotenv import load_dotenv

    load_dotenv()  # reads .env from the notebook's directory

    GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
    ENTREZ_EMAIL = os.getenv("ENTREZ_EMAIL")
    NCBI_API_KEY = os.getenv("NCBI_API_KEY")

    print("Running locally.")

# Validate required secrets
assert GEMINI_API_KEY, (
    "GEMINI_API_KEY is not configured -- get a free key at https://aistudio.google.com/api-keys"
)

assert ENTREZ_EMAIL, (
    "ENTREZ_EMAIL is not configured."
)

if NCBI_API_KEY:
    print("NCBI_API_KEY found.")
else:
    print("No NCBI_API_KEY found -- using the slower NCBI rate limit.")


## b) Fetch workshop data files (hash-verified download)

The triage flowchart/prompts and reference data are downloaded from GitHub and checked against a known checksum before use; all codes are in Python, so it runs standalone in Google Colab (or any Jupyter environment) with only a couple of `pip install`s.

The downloaded flowchart/prompts files use a JSON schema for the prompts and the flow to guide the language model.

In [ ]:
import hashlib
from pathlib import Path

import requests

# Set the base url from where the files will be downloaded
# GitHub example (pin to a commit SHA, not a branch, so the content can't change under you):
#   BASE_URL = "https://raw.githubusercontent.com/<org>/<repo>/<commit-sha>/workshop/release_data"
BASE_URL = "https://raw.githubusercontent.com/mdrishti/amrCourse/main/release_data"

# (filename, expected sha256) — computed once from the published files.
# If you edit any released file, recompute with: shasum -a 256 <file>
WORKSHOP_FILES = [
    ("flowchart_triage_amr_workshop.json",
     "a32faf064f0b9f51ca3ef813dac2694ff9a09b7f2548397430d37dff3ad19714"),
    ("prompts_triage_amr_workshop.json",
     "543f0ed53ac95758666b97a899b9ba5263af79bd57fba854d63ecb01c11e3076"),
    ("amr_phrase_candidates.csv",
     "fae322a7ffccb84afa7aa91cb0910d025914edd1463b177e664470806ef0ae6a"),
    ("word_histogram.csv",
     "f52e39f54714807eee093e2021e8b71e0afa1b70e7b12ca0b46a13e550038d5b"),
    ("amr_pmids_unique.csv",
    "9ccf61bc9ed19a70ddd1c0bcf055c1f3506f6c07ad2a0ea9a46338922fe8a0b5")
]

DATA_DIR = Path("workshop_data")
DATA_DIR.mkdir(exist_ok=True)


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(65536), b""):
            h.update(block)
    return h.hexdigest()


def fetch_verified(filename, expected_sha256, base_url=BASE_URL, dest_dir=DATA_DIR):
    out_path = dest_dir / filename  # fixed subdirectory — never a path from remote input
    url = f"{base_url.rstrip('/')}/{filename}"

    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    out_path.write_bytes(resp.content)

    actual = _sha256(out_path)
    if actual != expected_sha256:
        out_path.unlink(missing_ok=True)
        raise ValueError(
            f"Checksum mismatch for {filename}: expected {expected_sha256}, got {actual}. "
            "Refusing to use this file — re-check BASE_URL or the file on the host."
        )
    print(f"\u2713 {filename} downloaded and checksum-verified ({len(resp.content)} bytes)")
    return out_path


if BASE_URL.startswith("<<"):
    print(
        "BASE_URL is not set — skipping download. Falling back to the built-in minimal "
        "triage schema in Section 2 so the rest of the notebook still runs."
    )
    downloaded = {}
else:
    downloaded = {name: fetch_verified(name, sha) for name, sha in WORKSHOP_FILES}

## c) Where the corpus comes from: query design before triage

Before any LLM triage runs, someone had to decide which papers to pull from PubMed/PMC in the first place. That decision — the search query and year range — is a data-quality lever in its own right, upstream of and arguably as consequential as which LLM you triage with.



In [ ]:
AMR_TERMS = (
    '("antimicrobial resistance"[Title/Abstract] OR "antibiotic resistance"[Title/Abstract] '
    'OR "drug resistance, microbial"[MeSH Terms] OR "multidrug resistance"[Title/Abstract] '
    'OR "multidrug resistant"[Title/Abstract] OR "resistance gene"[Title/Abstract] OR "resistant gene"[Title/Abstract] OR "resistant genes"[Title/Abstract] '
    'OR "resistance genes"[Title/Abstract] OR "antimicrobial susceptibility"[Title/Abstract])'
)
#OR "beta-lactamase"[Title/Abstract] OR "efflux pump"[Title/Abstract] OR "minimum inhibitory concentration"[Title/Abstract] '


PUBTYPE_EXCLUSIONS = (
    'NOT "review"[pt] NOT "case reports"[pt] NOT "letter"[pt] NOT "comment"[pt] '
    'NOT "address"[pt] NOT "autobiography"[pt] NOT "biography"[pt] '
    'NOT "conference proceedings"[pt] NOT "editorial"[pt] NOT "interactive tutorial"[pt] '
    'NOT "interview"[pt] NOT "introductory journal article"[pt] '
    'NOT "personal narrative"[pt] NOT "portrait"[pt] NOT "news"[pt]'
)

# Optional terms to include
VARIANT_EXPERIMENT_TERMS = (
    '(("mutation"[MeSH Terms] OR "mutation"[All Fields] OR "mutations"[All Fields]) '
    'OR mutational[All Fields] OR mutant[All Fields] OR mutants[All Fields] '
    'OR variant[All Fields] OR variants[All Fields]) '
    'AND (experiment[All Fields] OR experimental[All Fields] OR "gene knockout"[All Fields] '
    'OR "site-directed mutagenesis"[All Fields] OR "functional analysis"[All Fields] '
    'OR "phenotypic characterization"[All Fields] OR "gene deletion"[All Fields] '
    'OR "insertional mutagenesis"[All Fields] OR "reverse genetics"[All Fields] '
    'OR "complementation assay"[All Fields])'
)

full_query = f'{AMR_TERMS} AND "pubmed pmc"[sb] {PUBTYPE_EXCLUSIONS}'

print(full_query)

**Discuss before running anything**: if you were building a corpus for your own project today, how many years back would you go — 5, 10, 25? Trade-offs:
- **Recall vs. noise** — wider ranges catch more papers but pull in more studies using outdated nomenclature/methods.
- **PMC/BioC availability** — `"pubmed pmc"[sb]` restricts to records with PMC full text at all; very old papers are disproportionately PubMed-only (abstract, no full text).
- **Publication-type drift** — case reports and reviews are a bigger share of older AMR literature.
- **Your research question** — tracking how resistance nomenclature evolved argues for a wide range *deliberately*; building an eval set for a current pipeline argues for a narrower, more homogeneous one.

In [ ]:
from Bio import Entrez

Entrez.email = ENTREZ_EMAIL
if NCBI_API_KEY:
    Entrez.api_key = NCBI_API_KEY

# NCBI rate limit: ~3 req/s without an API key, ~10 req/s with one (see Section 0's secrets cell).
NCBI_SLEEP = 0.11 if NCBI_API_KEY else 0.4

# Narrow YEAR RANGE + small retmax: this is a workshop-scale demo query, not a full corpus pull.
start_year, end_year = 2015, 2023
demo_retmax = 20

handle = Entrez.esearch(
    db="pubmed",
    term=full_query,
    datetype="PDAT",
    mindate=str(start_year),
    maxdate=str(end_year),
)
    #retmax=demo_retmax,

search_result = Entrez.read(handle)
handle.close()

demo_pmids = search_result["IdList"]
print(f"Query matched {search_result['Count']} PubMed records in {start_year}-{end_year} "
      f"(fetched first {len(demo_pmids)}):")
print(demo_pmids)

## The query may not be right the first time. Test out a few different terms in the query.

Priot to the course, `AMR_TERMS` was checked against a trusted reference set (the terms from CABBAGE paper) to see what the query missed. Missed PMIDs were bucketed into: no PMC full text at all / PMC exists but no `AMR_TERMS` match (the interesting bucket — real AMR papers the query's *wording* missed) / other (excluded by the mutation or publication-type filters).

For the "no term match" bucket, titles+abstracts were profiled two ways: a plain word-frequency histogram, and a 2–5 word n-gram scan ranked by how many distinct missed papers contain each phrase. These are the real, full results from that investigation — downloaded and checksum-verified in Section 0.0 above, not pasted inline.

In [ ]:
import pandas as pd

# Minimal built-in fallback (top 20 only) in case BASE_URL wasn't set in Section 0.0 —
# the full CSVs from the downloaded files are used preferentially when available.
_phrase_fallback = pd.DataFrame(
    [
        ("whole genome", 12, 18), ("genome sequencing", 11, 17),
        ("neisseria meningitidis", 11, 14), ("whole genome sequencing", 9, 15),
        ("sequence types", 9, 11), ("n gonorrhoeae", 8, 15),
        ("neisseria gonorrhoeae", 8, 15), ("invasive meningococcal", 7, 9),
        ("invasive meningococcal disease", 7, 8), ("meningococcal disease", 7, 8),
    ],
    columns=["phrase", "doc_count", "total_count"],
)

if "amr_phrase_candidates.csv" in downloaded:
    print("file found")
    phrase_candidates = pd.read_csv(downloaded["amr_phrase_candidates.csv"])
    word_histogram = pd.read_csv(downloaded["word_histogram.csv"])
else:
    print("amr_phrase_candidates.csv wasn't downloaded (BASE_URL not set) — skipping.")
    phrase_candidates = _phrase_fallback
    word_histogram = None

print("Top candidate phrases from papers the query missed (by how many distinct missed papers contain them):")
phrase_candidates.head(50)

**Discuss**: scan the top phrase candidates above. Which look like genuine AMR terminology worth adding as a new `OR "..."[Title/Abstract]` clause to `AMR_TERMS`, and which look like generic scientific-writing noise (organism names, methods language)? This is the same harmonisation/QC discipline you'll apply later to triage output itself — checking a systematic process against a trusted reference and feeding discrepancies back into the process.

### d) Excluding retracted papers and papers that refer to a retraction

`esearch` alone has no clean way to exclude retractions and linked Expression-of-Concern notices in one
query-level term, so this does it as a **second pass** over full records fetched with `efetch`:

1. For every matched PMID, fetch `PublicationTypeList` and `CommentsCorrectionsList`.
2. Flag a PMID as "drop" if its own `PublicationType` includes `Retracted Publication` / `Retraction Notice` /
   `Expression of Concern` — the record is itself a retraction/EoC notice.
3. Also flag any PMID *referenced* by such a record via `CommentsCorrections/@RefType` of `RetractionOf`,
   `RetractionIn`, `ExpressionOfConcernFor`, or `ExpressionOfConcernIn` — this is the paper **being**
   retracted/flagged, which may not say so itself in `PublicationType`.
4. Drop the union of both sets from the result.

This is implemented below in Python (`efetch` + `Entrez.read`), batched to stay under NCBI's URL-length
limits for large PMID lists.

In [ ]:
# NOTE: the following cell runs ~len(all_pmids)/200 efetch calls at NCBI_SLEEP seconds each
# for the full 56k-record corpus that's still a long time even with an API key.
# set the flag below to run on all  (`True`) or randomly picked 200 PMIDs (`False`)
RUN_ON_FULL_SET = False  # set True to check every one of the `all_pmids` records

In [ ]:
import time
from datetime import date, timedelta
from urllib.error import HTTPError, URLError
from http.client import IncompleteRead

RETRACTION_TYPES = {"Retracted Publication", "Retraction Notice", "Expression of Concern"}
RETRACTION_REFTYPES = {"RetractionOf", "RetractionIn", "ExpressionOfConcernFor", "ExpressionOfConcernIn"}


def entrez_call(fn, *args, max_retries=5, **kwargs):
    """Runs an Entrez.esearch/efetch call + Entrez.read with retry/backoff.

    NCBI connections occasionally drop mid-response on long-running loops (chunked-transfer
    read errors, HTTP 429/5xx) -- these are transient, not query bugs, so retry instead of
    letting one flaky request kill a multi-hour fetch."""
    for attempt in range(max_retries):
        try:
            handle = fn(*args, **kwargs)
            result = Entrez.read(handle)
            handle.close()
            return result
        except (HTTPError, URLError, IncompleteRead, ValueError) as e:
            wait = 2.0 * (2 ** attempt)
            print(f"  Entrez call failed ({e!r}), retrying in {wait:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(wait)
    raise RuntimeError(f"Entrez call failed after {max_retries} retries: {fn} {args} {kwargs}")


def find_retracted_pmids(pmids, batch_size=200):
    """Flags any PMID that is itself a retraction/Expression-of-Concern notice, or that is
    referenced by one (i.e. the paper being retracted/flagged, which may not say so in its
    own PublicationType)."""
    drop = set()

    for i in range(0, len(pmids), batch_size):
        batch = pmids[i:i + batch_size]
        records = entrez_call(Entrez.efetch, db="pubmed", id=batch, rettype="xml", retmode="xml")

        for article in records["PubmedArticle"]:
            citation = article["MedlineCitation"]
            pmid = str(citation["PMID"])

            pub_types = {str(t) for t in citation["Article"].get("PublicationTypeList", [])}
            if pub_types & RETRACTION_TYPES:
                drop.add(pmid)

            for correction in article.get("PubmedData", {}).get("CommentsCorrectionsList", []):
                ref_type = correction.attributes.get("RefType")
                if ref_type in RETRACTION_REFTYPES:
                    drop.add(pmid)  # this record itself refers to a retraction/EoC
                    linked_pmid = correction.get("PMID")
                    if linked_pmid:
                        drop.add(str(linked_pmid))  # the paper being retracted/flagged

        time.sleep(NCBI_SLEEP)  # ~3 req/s anonymous, ~10 req/s with NCBI_API_KEY set
        if (i // batch_size) % 10 == 0:
            print(f"  ...checked {min(i + batch_size, len(pmids))}/{len(pmids)}")

    return drop


# PubMed's esearch hard-caps retstart at 9998 -- there is NO way to page past the first 9,999
# records of a single query via retstart/WebEnv (NCBI's own docs point this out and recommend
# EDirect, which works around it by date-bisecting). We do the same thing here: recursively
# split the date range into smaller windows until each window's match count is <=9999, then
# union the PMIDs across windows. No retstart paging is used at all.
ESEARCH_CAP = 9999


def fetch_pmids_in_range(term, start_date, end_date, depth=0):
    result = entrez_call(
        Entrez.esearch,
        db="pubmed",
        term=term,
        datetype="PDAT",
        mindate=start_date.strftime("%Y/%m/%d"),
        maxdate=end_date.strftime("%Y/%m/%d"),
        retmax=ESEARCH_CAP,
    )
    time.sleep(NCBI_SLEEP)

    count = int(result["Count"])
    if count <= ESEARCH_CAP:
        return result["IdList"]

    if start_date >= end_date:
        # Can't split a single day any further -- NCBI's cap means some records on this
        # exact day are unreachable via esearch. Rare for a title/abstract query; flag it.
        print(f"WARNING: {count} records on {start_date} alone exceed the {ESEARCH_CAP} cap; "
              f"only the first {ESEARCH_CAP} for that day are included.")
        return result["IdList"]

    mid_date = start_date + (end_date - start_date) // 2
    print(f"{'  ' * depth}{start_date}..{end_date}: {count} hits > cap, splitting at {mid_date}")
    left = fetch_pmids_in_range(term, start_date, mid_date, depth + 1)
    right = fetch_pmids_in_range(term, mid_date + timedelta(days=1), end_date, depth + 1)
    return left + right


print(f"Fetching all PMIDs for {full_query!r} in {start_year}-{end_year} via date-bisection...")
all_pmids = fetch_pmids_in_range(full_query, date(start_year, 1, 1), date(end_year, 12, 31))
all_pmids = list(dict.fromkeys(all_pmids))  # de-dup while preserving order (adjacent windows can't overlap, but be safe)
print(f"Collected {len(all_pmids)} PMIDs total.")


import random
pmids_to_check = all_pmids if RUN_ON_FULL_SET else random.sample(all_pmids, min(200, len(all_pmids)))

print(f"Checking {len(pmids_to_check)} PMIDs for retraction / Expression-of-Concern status...")
retracted_or_flagged = find_retracted_pmids(pmids_to_check)
clean_pmids = [p for p in pmids_to_check if p not in retracted_or_flagged]

print(f"Dropped {len(retracted_or_flagged)} retracted/Expression-of-Concern PMIDs "
      f"({len(pmids_to_check)} checked -> {len(clean_pmids)} clean).")
if retracted_or_flagged:
    print("Dropped PMIDs:", sorted(retracted_or_flagged))

# Optional check

How many of the PMIDs covered in the AMR portal are present in this list of PMIDs obtained after all checks.

In [ ]:
import pandas as pd
from Bio import Entrez
import os

# 1. Load the PMIDs from amr_pmids_unique.csv inside the workshop_data folder
csv_path = "workshop_data/amr_pmids_unique.csv"
if os.path.exists(csv_path):
    csv_df = pd.read_csv(csv_path)
    print(f"Loaded {len(csv_df)} records from {csv_path}.")

    # Get unique PMIDs from CSV as strings
    csv_pmids = [str(x) for x in csv_df.iloc[:, 0].dropna().unique()]

    # 2. Check which CSV PMIDs match our search query in PubMed
    # We do this by searching for the PMIDs AND the search query in batches to avoid URL length limits
    print("Checking which PMIDs from the CSV match the search query...")
    matching_pmids = []
    batch_size = 300  # Safe batch size for URL queries

    for i in range(0, len(csv_pmids), batch_size):
        pmid_batch = csv_pmids[i:i + batch_size]
        pmid_query = " OR ".join([f"{pmid}[pmid]" for pmid in pmid_batch])
        combined_query = f"({full_query}) AND ({pmid_query})"

        try:
            handle = Entrez.esearch(
                db="pubmed",
                term=combined_query,
                datetype="PDAT",
                mindate=str(start_year),
                maxdate=str(end_year),
                retmax=batch_size
            )
            search_results = Entrez.read(handle)
            handle.close()
            matching_pmids.extend(search_results["IdList"])
        except Exception as e:
            print(f"Error processing batch starting at index {i}: {e}")

    print(f"\nResults:\n{len(matching_pmids)} out of the {len(csv_df)} CSV PMIDs match the 2015-2022 PubMed query.")
    if len(matching_pmids) > 0:
        print("Sample of matching PMIDs:", matching_pmids[:10])
else:
    print(f"\nError: '{csv_path}' not found in the workshop_data directory.")

---
## e) The triage schema — downloaded, checksum-verified, same shape as production

`flowchart_triage_amr_workshop.json` / `prompts_triage_amr_workshop.json` were fetched and verified. Nodes in the JSON have `type`/`data.prompt_name`/`transitions`, prompts are `{name, prompt, output_format}`.



In [ ]:
import json

_FALLBACK_FLOWCHART = {
    "startNode": "check_bacteria",
    "nodes": {
        "check_bacteria": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "is_about_bacteria"},
            "transitions": {"true": "check_gene_genotype", "false": "terminal_triage"},
        },
        "check_gene_genotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_gene_genotype"},
            "transitions": {"true": "check_phenotype", "false": "check_phenotype"},
        },
        "check_phenotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_phenotype_data"},
            "transitions": {"true": "check_amr_phenotype", "false": "check_amr_phenotype"},
        },
        "check_amr_phenotype": {
            "type": "conditional_prompt_boolean",
            "data": {"prompt_name": "has_amr_phenotype"},
            "transitions": {"true": "terminal_triage", "false": "terminal_triage"},
        },
        "terminal_triage": {"type": "terminal_triage", "data": {}, "transitions": {}},
    },
}
_FALLBACK_PROMPTS = {
    "prompts": [
        {"name": "is_about_bacteria",
         "prompt": "Does this paper study bacteria or bacterial strains?"},
        {"name": "has_gene_genotype",
         "prompt": ("Does this paper report a specific bacterial gene or genotype by name (e.g. a named "
                     "resistance gene, efflux pump gene, or genotype/lineage designation) in the bacterial "
                     "strain(s) under study, even if no specific sequence variant/mutation is described? "
                     "Answer NO if no gene or genotype is named for the study bacterium.")},
        {"name": "has_phenotype_data",
         "prompt": ("Does this paper report any experimental measurements or observations of "
                     "bacterial phenotypes (e.g. growth, metabolism, resistance, biofilm formation, virulence)?")},
        {"name": "has_amr_phenotype",
         "prompt": ("Does this paper report a phenotype specifically related to antimicrobial/antibiotic "
                     "resistance or susceptibility for the bacterial strain(s) under study (e.g. MIC values, "
                     "resistant/susceptible/intermediate classification, resistance gene expression, inhibition "
                     "zone diameter)? Answer NO if the only phenotypes reported are unrelated to antimicrobial "
                     "resistance/susceptibility (e.g. growth rate, biofilm formation, virulence with no resistance angle).")},
    ],
}

if "flowchart_triage_amr_workshop.json" in downloaded:
    TRIAGE_FLOWCHART = json.loads(downloaded["flowchart_triage_amr_workshop.json"].read_text())
    prompts_raw = json.loads(downloaded["prompts_triage_amr_workshop.json"].read_text())
else:
    TRIAGE_FLOWCHART = _FALLBACK_FLOWCHART
    prompts_raw = _FALLBACK_PROMPTS

TRIAGE_PROMPTS = {p["name"]: p for p in prompts_raw["prompts"]}

print("Start node:", TRIAGE_FLOWCHART["startNode"])
print("Questions:", list(TRIAGE_PROMPTS.keys()))

## f) The Gemini call to triage

`call_llm` sends one prompt+paper-text pair to the Gemini API and parses the `{"answer": ..., "reasoning": ...}` JSON out of the response. `run_triage` walks `TRIAGE_FLOWCHART` node by node: at each `conditional_prompt_boolean` node call the LLM and branch on `true`/`false`, stop at `terminal_triage`.

In [ ]:
# SET the MODEL NAME you would like to use via the Gemini API, and its free-tier
# RPM (requests/minute) and RPD (requests/day) caps -- check current limits at
# https://ai.google.dev/gemini-api/docs/rate-limits if this model's caps change.
GEMINI_URL_TEMPLATE = "https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent"
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_RPM = 15
GEMINI_RPD = 500

MIN_REQUEST_INTERVAL = 60.0 / GEMINI_RPM  # spaced out evenly across a minute
_last_call_end = 0.0
_daily_request_count = 0


In [ ]:
import re
import time

# Gemini can return 429 (per-minute/per-day quota) or a transient 5xx -- both are worth
# retrying with backoff rather than failing the whole triage run on one blip.
RETRYABLE_STATUS_CODES = {429, 500, 502, 503}


def _extract_json(text):
    """Model output is usually pure JSON, but strip stray markdown fences/prose defensively."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(match.group(0)) if match else {"answer": "no", "reasoning": f"unparsed: {text[:200]}"}


def call_llm(question_prompt, paper_text, max_retries=5):
    global _last_call_end, _daily_request_count

    if _daily_request_count >= GEMINI_RPD:
        raise RuntimeError(
            f"Reached the configured daily cap (GEMINI_RPD={GEMINI_RPD}) for this session -- "
            "Gemini would reject further requests anyway. Wait until tomorrow, raise GEMINI_RPD "
            "if your actual quota is higher, or switch to the local model (section i)."
        )

    wait = MIN_REQUEST_INTERVAL - (time.time() - _last_call_end)
    if wait > 0:
        time.sleep(wait)

    user_content = (
        f"{question_prompt}\n\n"
        f"Paper text:\n{paper_text[:6000]}\n\n"
        'Respond ONLY with JSON in exactly this shape: {"answer": "yes" or "no", "reasoning": "one sentence"}'
    )

    url = GEMINI_URL_TEMPLATE.format(model=GEMINI_MODEL)

    for attempt in range(max_retries):
        resp = requests.post(
            url,
            params={"key": GEMINI_API_KEY},
            json={
                "contents": [{"parts": [{"text": user_content}]}],
                "generationConfig": {"temperature": 0},
            },
            timeout=120,
        )
        _last_call_end = time.time()
        _daily_request_count += 1

        if resp.status_code in RETRYABLE_STATUS_CODES:
            backoff = 5.0 * (2 ** attempt)
            print(f"  {resp.status_code} ({resp.text[:200]!r}), backing off {backoff:.0f}s "
                  f"(attempt {attempt + 1}/{max_retries})...")
            time.sleep(backoff)
            continue

        if not resp.ok:
            # Surface the actual reason (bad model id, invalid key, etc.) instead of a bare
            # HTTPError -- Gemini's error body usually says exactly what's wrong.
            print(f"  Gemini error {resp.status_code}: {resp.text[:500]}")
            resp.raise_for_status()

        content = resp.json()["candidates"][0]["content"]["parts"][0]["text"]
        parsed = _extract_json(content)
        answer = str(parsed.get("answer", "")).strip().lower() == "yes"
        return answer, parsed.get("reasoning", "")

    raise RuntimeError(
        "Gemini call failed after retries -- likely your free-tier rate limit (requests/minute "
        "or requests/day). Wait a bit and retry."
    )


def run_triage(paper_text, verbose=True):
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        question_prompt = TRIAGE_PROMPTS[prompt_name]["prompt"]

        answer, reasoning = call_llm(question_prompt, paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_phenotype": extraction_data.get("has_phenotype_data"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


## g) Get sample papers

The pipeline triages full-text BioC sections; for this self-contained demo we fetch title+abstract from PubMed for a handful of known AMR papers via `Bio.Entrez` — a reasonable simplification for a workshop exercise, not what the production pipeline does (it sees full text, and uses PMC's BioC XML/JSON export, not just the abstract).

In [ ]:
# A small curated set of real PMIDs — a mix of clear AMR papers and one that isn't AMR-focused,
# so triage has something to actually discriminate on.
#sample_pmids = ["12737743", "12800222", "12927083", "12059906"]
sample_pmids = ['12737743', '12800222', '12927083', '12059906','34515028', '34485958', '34463568', '34461978', '34414415'] # these are reported in AMR portal


def fetch_title_abstract(pmid):
    handle = Entrez.efetch(db="pubmed", id=pmid, rettype="abstract", retmode="xml")
    records = Entrez.read(handle)
    handle.close()
    article = records["PubmedArticle"][0]["MedlineCitation"]["Article"]
    title = str(article.get("ArticleTitle", ""))
    abstract_parts = article.get("Abstract", {}).get("AbstractText", [])
    abstract = "\n".join(str(p) for p in abstract_parts)
    return f"{title}\n\n{abstract}"


papers = {}
for pmid in sample_pmids:
    papers[pmid] = fetch_title_abstract(pmid)
    time.sleep(NCBI_SLEEP)

print(f"Fetched {len(papers)} papers")
print(papers[sample_pmids[0]][:400])

## Run triage on title/abstract across several papers and tabulate

**Watch the free-tier cap**: Gemini's free tier is rate-limited per-minute and per-day (check current limits at https://ai.google.dev/gemini-api/docs/rate-limits) -- this loop uses up to 4 requests per paper (one per triage question), so don't run it against more than a handful of papers in one round.

In [ ]:
import requests

rows = []
for pmid, text in papers.items():
    print(f"Triaging PMID {pmid}...")
    try:
        result = run_triage(text, verbose=False)
        rows.append({"pmid": pmid, **result})
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] Failed to triage PMID {pmid} due to network/API error: {e}")

df = pd.DataFrame(rows)
df

---
## h) Triage full-text of the papers, section-aware

So far triage has run on title+abstract only, as one blob of text per question.For many of these papers, you would see that the pipeline fails at the first step (`is_about_bacteria`). If you will read the abstract, you will realize that these papers are in fact are about bacteria and the triage did not work as expected.

To improve the triage, instead routes each triage question to the full text instead of only title+abstract. Tailor the prompts to specific the paper section(s) listed in its `target_section`
(IAO ontology codes — see `TRIAGE_PROMPTS[...]["target_section"]` loaded in section (e)), using the full text of the paper, not just the abstract.

This section fetches a paper's full text directly from PMC as JATS XML (`Entrez.efetch(db="pmc", ...)`)
and splits it into `{iao_id: text}` by section heading — entirely self-contained, using only the NCBI
APIs this notebook already depends on. **This only works for papers in PMC's open-access full-text
subset** — not every PMID has one; a PMID with no PMC full-text link, or a PMC record that's
metadata-only, will come back with few or no sections below (title/abstract may still work via the
`is_about_bacteria`/`has_gene_genotype` fallback to whatever sections were found).

In [ ]:
import xml.etree.ElementTree as ET

# Same vocabulary as the target_section IAO codes in TRIAGE_PROMPTS (section e).
SECTION_HEADING_TO_IAO = {
    "abstract": "IAO:0000315",
    "introduction": "IAO:0000316", "intro": "IAO:0000316", "background": "IAO:0000316",
    "methods": "IAO:0000317", "materials and methods": "IAO:0000317",
    "results": "IAO:0000318",
    "discussion": "IAO:0000319", "discuss": "IAO:0000319",
}


def pmid_to_pmcid(pmid):
    """Look up the PMC ID (if any) linked to a PubMed PMID -- not every paper
    has one; this returns None when there's no PMC full-text record."""
    handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
    result = Entrez.read(handle)
    handle.close()
    try:
        return f"PMC{result[0]['LinkSetDb'][0]['Link'][0]['Id']}"
    except (IndexError, KeyError):
        return None


def fetch_sections_from_pmc(pmcid):
    """Fetch a PMC open-access full-text article as JATS XML via Entrez and split
    it into {iao_id: text} -- entirely self-contained (just NCBI + Bio.Entrez,
    no local files or other repos needed). Only works for PMC's open-access
    full-text subset; returns whatever sections it can find (possibly just
    title/abstract, or an empty dict for a metadata-only / closed-access record)."""
    handle = Entrez.efetch(db="pmc", id=pmcid, rettype="full", retmode="xml")
    xml_text = handle.read()
    handle.close()
    root = ET.fromstring(xml_text)

    sections = {}

    title_el = root.find(".//article-title")
    if title_el is not None:
        sections["IAO:0000305"] = "".join(title_el.itertext()).strip()

    abstract_el = root.find(".//abstract")
    if abstract_el is not None:
        sections["IAO:0000315"] = "".join(abstract_el.itertext()).strip()

    for sec in root.findall(".//body//sec"):
        heading_el = sec.find("title")
        heading = (heading_el.text or "").strip().lower() if heading_el is not None else ""
        sec_type = (sec.get("sec-type") or "").strip().lower()
        key = sec_type or heading
        iao_id = next((v for name, v in SECTION_HEADING_TO_IAO.items() if name in key), None)
        if not iao_id:
            continue
        text = "".join(sec.itertext()).strip()
        sections[iao_id] = (sections.get(iao_id, "") + "\n" + text).strip()

    return sections


def select_target_text(sections, target_section_ids, max_chars=6000):
    """Concatenate only the sections a prompt cares about; fall back to
    everything available if none of its target sections were found."""
    matched = [sections[i] for i in target_section_ids if i in sections]
    text = "\n\n".join(matched) if matched else "\n\n".join(sections.values())
    return text[:max_chars]


def run_triage_sectioned(sections, verbose=True):
    """Same flowchart walk as run_triage, but each question only sees the
    section(s) listed in its own target_section instead of one flat blob."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        prompt_def = TRIAGE_PROMPTS[prompt_name]
        paper_text = select_target_text(sections, prompt_def.get("target_section", []))

        answer, reasoning = call_llm(prompt_def["prompt"], paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_phenotype": extraction_data.get("has_phenotype_data"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }

### Try it

Pick one or all of your `sample_pmids`, look up its PMC ID, and fetch+triage the full text.

In [ ]:
rows = []
for pmid, text in papers.items():
    print(f"Triaging PMID {pmid}...")
    pmcid = pmid_to_pmcid(pmid)
    
    if pmcid is None:
        print(f"PMID {pmid} has no linked PMC full-text record -- try a different PMID.")
    else:
        print(f"PMID {pmid} -> {pmcid}")
        sections = fetch_sections_from_pmc(pmcid)
        print("Sections found:", list(sections.keys()))
    
        if sections:
            result = run_triage_sectioned(sections)
            print("\nResult:", result)
            rows.append({"pmid": pmid, **result})
        else:
            print("No usable sections came back (metadata-only or closed-access record)")
df = pd.DataFrame(rows)
df

---
## i) Advanced: host an open weight model on Colab or your personal laptop

Everything so far has called the Gemini API, which is rate-limited on the free tier and depends on
a third party staying up. This section instead downloads a small open-weight model and runs it
**locally** -- inside this Colab VM, or on your own laptop if you're running the notebook there (see
"Running it locally" in the course README).

The next cell detects what you're running on and picks a backend automatically:
- **Apple Silicon Mac** (M-series chip) -> `mlx-lm`, Apple's native framework -- fastest on that hardware.
- **Anything else** (Colab, Linux, Intel Mac) -> `llama.cpp` with a GGUF model -- works everywhere,
  CPU-only is fine.

Qwen3-14B is the default below -- comfortable on 24GB+ RAM (e.g. an M4 Pro laptop) or Colab. Drop to
Qwen3-4B or Qwen3-8B (smaller, faster, slightly less sharp) if 14B is too slow on your machine.

In [ ]:
import platform

IS_APPLE_SILICON = platform.system() == "Darwin" and platform.machine() == "arm64"
LOCAL_BACKEND = "mlx" if IS_APPLE_SILICON else "llama_cpp"

print(f"Detected platform: {platform.system()} {platform.machine()} -> using the {LOCAL_BACKEND!r} backend")


In [ ]:
if LOCAL_BACKEND == "mlx":
    !pip install -q mlx-lm
    MLX_MODEL_REPO = "mlx-community/Qwen3-14B-4bit"  # swap to a smaller mlx-community Qwen3 repo if this is too slow
else:
    !pip install -q llama-cpp-python huggingface_hub
    from huggingface_hub import list_repo_files

    LOCAL_MODEL_REPO = "Qwen/Qwen3-14B-GGUF"  # swap to "Qwen/Qwen3-8B-GGUF" or "Qwen/Qwen3-4B-GGUF" for a smaller/faster model
    print(f"GGUF files available in {LOCAL_MODEL_REPO}:")
    for f in list_repo_files(LOCAL_MODEL_REPO):
        if f.endswith(".gguf"):
            print(" ", f)


In [ ]:
if LOCAL_BACKEND == "llama_cpp":
    from huggingface_hub import hf_hub_download

    # Pick one of the filenames printed above -- Q4_K_M is the usual speed/quality sweet spot.
    LOCAL_MODEL_FILE = "Qwen3-14B-Q4_K_M.gguf"

    # hf_hub_download already checks the local HF cache and skips re-downloading if this file
    # was fetched before -- no separate try_to_load_from_cache check needed.
    model_path = hf_hub_download(repo_id=LOCAL_MODEL_REPO, filename=LOCAL_MODEL_FILE)
    print("Model path:", model_path)
else:
    print("Using MLX -- no separate download step; the model downloads inside load() below.")


In [ ]:
if LOCAL_BACKEND == "mlx":
    from mlx_lm import load as mlx_load

    mlx_model, mlx_tokenizer = mlx_load(MLX_MODEL_REPO)
    print(f"Loaded {MLX_MODEL_REPO} via MLX.")
else:
    import os
    from llama_cpp import Llama

    llm_local = Llama(
        model_path=model_path,
        n_ctx=4096,
        n_threads=os.cpu_count(),
        verbose=False,
    )
    print("Loaded via llama.cpp.")


In [ ]:
import re as _re_local


def strip_thinking(response):
    """Remove <think>...</think> blocks Qwen3 (and similar reasoning models) emit before
    their actual answer. Without this, the think block can quote the output-format template
    verbatim and corrupt the JSON match below; the /no_think suppression in _local_generate
    should normally prevent the block from appearing at all, but this is a defensive second
    layer in case a model still emits one."""
    cleaned = _re_local.sub(r"<think>.*?</think>", "", response, flags=_re_local.DOTALL)
    cleaned = _re_local.sub(r"<think>.*$", "", cleaned, flags=_re_local.DOTALL)
    return cleaned.strip()


def _local_generate(user_content, max_tokens=200):
    """Backend-agnostic text generation -- hides the mlx-lm vs llama.cpp difference from
    call_llm_local, so run_triage_local/run_triage_local_sectioned don't need to care which
    backend section (i) picked."""
    if LOCAL_BACKEND == "mlx":
        from mlx_lm import generate as mlx_generate

        messages = [{"role": "user", "content": user_content}]
        try:
            # enable_thinking=False is Qwen3's own chat-template switch to skip the
            # <think> block entirely -- cheaper and more reliable than stripping it after
            # the fact. Falls back to no kwarg if this tokenizer's template doesn't support it.
            prompt = mlx_tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, enable_thinking=False
            )
        except TypeError:
            prompt = mlx_tokenizer.apply_chat_template(messages, add_generation_prompt=True)
        # mlx-lm's generate()/sampler API has changed across versions -- if this errors with an
        # unexpected keyword, check `pip show mlx-lm` and the mlx-lm README for the current
        # signature (e.g. newer versions take a `sampler=` object instead of `temp=`).
        return mlx_generate(mlx_model, mlx_tokenizer, prompt=prompt, max_tokens=max_tokens, verbose=False)
    else:
        # /no_think is Qwen3's own in-text switch to skip the <think> block.
        resp = llm_local.create_chat_completion(
            messages=[{"role": "user", "content": user_content + " /no_think"}],
            temperature=0,
            max_tokens=max_tokens,
        )
        return resp["choices"][0]["message"]["content"]


def call_llm_local(question_prompt, paper_text, max_tokens=200):
    """Same prompt contract as call_llm (section f), answered by the local model instead."""
    user_content = (
        f"{question_prompt}\n\n"
        f"Paper text:\n{paper_text[:6000]}\n\n"
        'Respond ONLY with JSON in exactly this shape: {"answer": "yes" or "no", "reasoning": "one sentence"}'
    )
    content = _local_generate(user_content, max_tokens=max_tokens)
    content = strip_thinking(content)
    parsed = _extract_json(content)
    answer = str(parsed.get("answer", "")).strip().lower() == "yes"
    return answer, parsed.get("reasoning", "")


def run_triage_local(paper_text, verbose=True):
    """Same flowchart walk as run_triage (section f), using the local model via call_llm_local."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        question_prompt = TRIAGE_PROMPTS[prompt_name]["prompt"]

        answer, reasoning = call_llm_local(question_prompt, paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_phenotype": extraction_data.get("has_phenotype_data"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


def run_triage_local_sectioned(sections, verbose=True):
    """Section-aware version of run_triage_local -- mirrors run_triage_sectioned (section h),
    but answered by the local model via call_llm_local instead of Gemini. Each question
    only sees the section(s) listed in its own target_section, same as section h."""
    node_name = TRIAGE_FLOWCHART["startNode"]
    extraction_data = {}

    while True:
        node = TRIAGE_FLOWCHART["nodes"][node_name]
        if node["type"] == "terminal_triage":
            break

        prompt_name = node["data"]["prompt_name"]
        prompt_def = TRIAGE_PROMPTS[prompt_name]
        paper_text = select_target_text(sections, prompt_def.get("target_section", []))

        answer, reasoning = call_llm_local(prompt_def["prompt"], paper_text)
        extraction_data[prompt_name] = answer
        if verbose:
            print(f"  [{prompt_name}] {answer} — {reasoning}")

        node_name = node["transitions"]["true" if answer else "false"]

    return {
        "is_bacteria": extraction_data.get("is_about_bacteria"),
        "has_gene_genotype": extraction_data.get("has_gene_genotype"),
        "has_phenotype": extraction_data.get("has_phenotype_data"),
        "has_amr_phenotype": extraction_data.get("has_amr_phenotype"),
    }


### Try it

Run every paper in `sample_pmids` through your local model. `USE_FULL_TEXT` below controls which
input it sees -- `True` (default) fetches full text and goes section-aware using the same
`fetch_sections_from_pmc`/`select_target_text` machinery from section (h); `False` uses title+abstract
only, same as `run_triage`/`run_triage_local` elsewhere. This only needs the local model loaded above;
no Gemini API call is involved.

In [ ]:
# Set if you would like to run triage on title+abstract only or full-text
USE_FULL_TEXT = False  # False -> title+abstract only (same input run_triage/run_triage_local use
                      # elsewhere); True -> full text, section-aware (section h's approach)

In [ ]:
def triage_local_for_pmid(pmid, use_full_text=USE_FULL_TEXT):
    """Triage one PMID with the local model only, either title+abstract or full-text
    section-aware depending on use_full_text."""
    if not use_full_text:
        if pmid not in papers:
            print(f"PMID {pmid}: not in `papers` (fetched in section g) -- skipping.")
            return {"pmid": pmid, "pmcid": None, "skipped_reason": "no abstract text available"}
        result = run_triage_local(papers[pmid], verbose=False)
        row = {"pmid": pmid, "pmcid": None, "skipped_reason": None}
        row.update(result)
        return row

    pmcid = pmid_to_pmcid(pmid)
    if pmcid is None:
        print(f"PMID {pmid}: no linked PMC full-text record -- skipping.")
        return {"pmid": pmid, "pmcid": None, "skipped_reason": "no PMC full-text record"}

    sections = fetch_sections_from_pmc(pmcid)
    if not sections:
        print(f"PMID {pmid} -> {pmcid}: no usable sections (metadata-only/closed-access) -- skipping.")
        return {"pmid": pmid, "pmcid": pmcid, "skipped_reason": "no usable sections"}

    print(f"PMID {pmid} -> {pmcid}, sections found: {list(sections.keys())}")
    result = run_triage_local_sectioned(sections, verbose=False)

    row = {"pmid": pmid, "pmcid": pmcid, "skipped_reason": None}
    row.update(result)
    return row


local_rows = [triage_local_for_pmid(pmid) for pmid in sample_pmids]
local_df = pd.DataFrame(local_rows)
local_df

# To try just one PMID instead of looping over all of sample_pmids:
# triage_local_for_pmid(sample_pmids[0])


### Optional: compare against Gemini

Check how often the local model agrees with the remote Gemini model on the same input. Uses whichever
mode `USE_FULL_TEXT` is set to above (full text, section-aware, or title+abstract only) for both
models, so the comparison stays apples-to-apples. Skipped by default -- set
`RUN_REMOTE_COMPARISON = True` below to run it. This needs a working `GEMINI_API_KEY` and model (see
section f); if an individual call fails (rate limit, model unavailable, etc.) that paper's remote
result is skipped rather than stopping the whole comparison.

In [ ]:
# OPTIONAL

RUN_REMOTE_COMPARISON = False  # set True to also call Gemini and compare against it


def compare_remote_vs_local(pmid, use_full_text=USE_FULL_TEXT):
    """Run one PMID through both models -- either title+abstract or full-text section-aware,
    matching whichever mode USE_FULL_TEXT (set above triage_local_for_pmid) selects -- and
    return one row of results (or a row noting why it was skipped)."""
    if not use_full_text:
        if pmid not in papers:
            print(f"PMID {pmid}: not in `papers` (fetched in section g) -- skipping.")
            return {"pmid": pmid, "pmcid": None, "skipped_reason": "no abstract text available"}

        paper_text = papers[pmid]
        result_remote = None
        try:
            result_remote = run_triage(paper_text, verbose=False)
        except Exception as e:
            print(f"  [Skipping Gemini] Remote call failed: {e}\n"
                  f"  (Likely you hit the free-tier rate limit -- wait a bit and retry.)")
        result_local = run_triage_local(paper_text, verbose=False)
        pmcid = None
    else:
        pmcid = pmid_to_pmcid(pmid)
        if pmcid is None:
            print(f"PMID {pmid}: no linked PMC full-text record -- skipping.")
            return {"pmid": pmid, "pmcid": None, "skipped_reason": "no PMC full-text record"}

        sections = fetch_sections_from_pmc(pmcid)
        if not sections:
            print(f"PMID {pmid} -> {pmcid}: no usable sections (metadata-only/closed-access) -- skipping.")
            return {"pmid": pmid, "pmcid": pmcid, "skipped_reason": "no usable sections"}

        print(f"PMID {pmid} -> {pmcid}, sections found: {list(sections.keys())}")

        result_remote = None
        try:
            result_remote = run_triage_sectioned(sections, verbose=False)
        except Exception as e:
            print(f"  [Skipping Gemini] Remote call failed: {e}\n"
                  f"  (Likely you hit the free-tier rate limit -- wait a bit and retry.)")
        result_local = run_triage_local_sectioned(sections, verbose=False)

    row = {"pmid": pmid, "pmcid": pmcid, "skipped_reason": None}
    for key, value in result_local.items():
        row[f"{key}_local"] = value
    if result_remote is not None:
        for key, value in result_remote.items():
            row[f"{key}_remote"] = value
        row["agree_on_every_question"] = (result_remote == result_local)
    else:
        row["agree_on_every_question"] = None
    return row


if RUN_REMOTE_COMPARISON:
    compare_rows = [compare_remote_vs_local(pmid) for pmid in sample_pmids]
    compare_df = pd.DataFrame(compare_rows)
    compare_df
else:
    print("RUN_REMOTE_COMPARISON is False -- skipping the Gemini comparison. "
          "Set it to True above and re-run this cell to use it.")


---
## j) Clean up: free the local model's memory

The model loaded in section (i) (several GB, whichever backend you used) stays resident for the
rest of this kernel session otherwise. Run this once you're done triaging to free it up -- safe to
run even if you never used section (i) at all.

In [ ]:
import gc

if "llm_local" in globals():
    del llm_local
    print("Freed llama.cpp model.")

if "mlx_model" in globals():
    del mlx_model
    del mlx_tokenizer
    print("Freed MLX model.")

gc.collect()

if LOCAL_BACKEND == "mlx":
    try:
        import mlx.core as mx
        mx.clear_cache()
    except AttributeError:
        # Older mlx-core versions expose this under mx.metal instead.
        import mlx.core as mx
        mx.metal.clear_cache()

print("Done.")
